# 02b — Encounter features

Compute previous encounters and time since the last admission. Duplicate spells are reduced before feature generation.

In [ ]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
from pathlib import Path
from icare_risk.utils.paths import get_path, get_data_root_from_schema
from icare_risk.utils.preprocessing import keep_first_spell

# Define schema configuration to use.
SCHEMA_CFG = 'configs/schema_1000p.yaml'
PHENOTYPES_CFG = 'configs/phenotypes.yaml'

# Paths
DATA_DIR = get_data_root_from_schema(schema_path=get_path(SCHEMA_CFG)) 
DATA_PROC_DIR = DATA_DIR / 'processed'
DATA_PROC_DIR.mkdir(parents=True, exist_ok=True)

# Debug
print(f"Using data from: {DATA_DIR}")

Using data from: /app/data/synthetic/10000p


In [3]:
# --- OPTION A: Load using pandas ---
from icare_risk.clinphen.io.loaders import load_from_pandas

# Load dataframes
dfs = load_from_pandas(
    schema_path=SCHEMA_CFG,
    domains_to_load=['episodes'],
    verbose=1
)

✅ Schema validation passed! All files exist and all physical columns match successfully.
Loaded... episodes (2.46MB) shape (10000, 11)


In [ ]:
# --- OPTION B: Load using DuckDB ---
from icare_risk.clinphen.io.loaders import load_from_duckdb

# Configure
domains = ['episodes']

# Load
dfs = load_from_duckdb(
    schema_path=SCHEMA_CFG,
    phenotype_path=PHENOTYPES_CFG,
    domains_to_load=domains,
    custom_codes={}, skip_mapping=True, 
    case_sensitive=False, verbose=1
)

# Assign
df_episodes = keep_first_spell(dfs['episodes'])

In [4]:
from icare_risk.clinphen.utils import (
    compute_previous_encounters,
    compute_time_since_last_admission
)

df_encounters = compute_previous_encounters(df_episodes, days=365)
df_encounters = compute_time_since_last_admission(
    df_episodes=df_encounters,
    subject_col="SUBJECT",
    spell_col="SPELL_IDENTIFIER",
    admission_col="ADMISSION_DATE",
    discharge_col="DISCHARGE_DATE",
    use_discharge_time=True,
)
df_encounters = keep_first_spell(df_encounters, name="encounter features")
display(df_encounters.head())


NameError: name 'df_episodes' is not defined

In [ ]:
filename = "02c_derived_features_encounters.parquet"
output_path = DATA_PROC_DIR / filename
df_encounters.to_parquet(output_path, index=False)
print(f"Saved: {output_path}")

NameError: name 'DATA_PROC_DIR' is not defined